# CodonFM (Encodon): codon-optimizing a real gene, round by round

![CodonFM (Encodon)](https://proto-bio.github.io/proto-assets/images/tool/codonfm/hero.png)

CodonFM / **Encodon** ([Darabi et al., 2025](https://research.nvidia.com/labs/dbr/assets/data/manuscripts/nv-codonfm-preprint.pdf)) is a bidirectional (masked) codon language model. We take a **real gene** (human preproinsulin, natural CDS from NCBI), tour every CodonFM tool, then **iteratively improve** the sequence with a CodonFM-guided *synonymous* codon optimization — one codon per round, re-scoring in context each time (the model is bidirectional, so every swap changes its neighbours' scores). We plot the full **progression** of every metric (CodonFM fitness, CAI, MFE, GC%, U%) across the rounds, and show that CodonFM's learned fitness already **co-varies with CAI and MFE**.

> **License:** Encodon code is Apache-2.0; weights are under the NVIDIA Open Model License. The `nvidia/NV-CodonFM-Encodon-*-v1` HuggingFace repos are public — no token required.

In [ ]:
from proto_tools.utils.notebook_docs import (
    display_api_reference,
    display_available_tools,
    display_doc_link,
    display_docs_section,
    display_overview,
)

display_doc_link("codonfm")
display_overview("codonfm")
display_docs_section("codonfm", "Background")

## Available tools

In [ ]:
display_available_tools("codonfm")

## The design target — human preproinsulin (natural CDS)

The wild-type coding sequence of human insulin (RefSeq NM_000207), fetched from NCBI.

In [ ]:
from itertools import product

DEVICE = "cuda"
CHECKPOINT = "encodon_80m"

_BASES = "TCAG"
_AAS = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
STANDARD_GENETIC_CODE = {a + b + c: aa for (a, b, c), aa in zip(product(_BASES, repeat=3), _AAS)}
SYNONYMS = {}
for _c, _aa in STANDARD_GENETIC_CODE.items():
    SYNONYMS.setdefault(_aa, []).append(_c)


def translate(cds):
    return "".join(STANDARD_GENETIC_CODE[cds[i : i + 3]] for i in range(0, len(cds), 3))


NATURAL_CDS = "ATGGCCCTGTGGATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGACCCAGCCGCAGCCTTTGTGAACCAACACCTGTGCGGCTCACACCTGGTGGAAGCTCTCTACCTAGTGTGCGGGGAACGAGGCTTCTTCTACACACCCAAGACCCGCCGGGAGGCAGAGGACCTGCAGGTGGGGCAGGTGGAGCTGGGCGGGGGCCCTGGTGCAGGCAGCCTGCAGCCCTTGGCCCTGGAGGGGTCCCTGCAGAAGCGTGGCATTGTGGAACAATGCTGTACCAGCATCTGCTCCCTCTACCAGCTGGAGAACTACTGCAACTAG"
PROTEIN = translate(NATURAL_CDS)
print(f"gene: human preproinsulin (INS)  |  {len(NATURAL_CDS)} nt / {len(NATURAL_CDS)//3} codons")
print(f"protein ({len(PROTEIN)} aa): {PROTEIN}")

## The five tools, one at a time

Each tool is shown on the same gene: its input, its config, the call, and what comes back.
The full codon-optimization study that uses them together comes after.

### `run_codonfm_fitness` — mean codon log-likelihood

Fitness scores a whole coding sequence: the mean log-likelihood Encodon assigns to its codons,
each predicted from the rest of the sequence. It is the objective the optimization later maximizes.


In [ ]:
from proto_tools import CodonFMFitnessConfig, CodonFMFitnessInput, run_codonfm_fitness


In [ ]:
# Display docs
display_api_reference("codonfm", "input", "run_codonfm_fitness")

# The natural human preproinsulin CDS defined above
fitness_input = CodonFMFitnessInput(sequences=[NATURAL_CDS])


In [ ]:
# Display config docs
display_api_reference("codonfm", "config", "run_codonfm_fitness")

fitness_config = CodonFMFitnessConfig(model_checkpoint=CHECKPOINT, device=DEVICE)


In [ ]:
# Run the fitness function
fitness_result = run_codonfm_fitness(fitness_input, fitness_config)


In [ ]:
# Display output docs
display_api_reference("codonfm", "output", "run_codonfm_fitness")

print(f"fitness (mean codon log-likelihood): {fitness_result.results[0].fitness:.4f}")


### `run_codonfm_score` — one codon substitution, reference vs alternative

Scoring compares two codons at one position and returns a log-likelihood ratio. Codon 2 of
preproinsulin is `GCC` (Ala); below it is compared against a synonymous swap and a missense one,
so the two cases can be read side by side.


In [ ]:
from proto_tools import CodonFMScoreConfig, CodonFMScoreInput, run_codonfm_score


In [ ]:
# Display docs
display_api_reference("codonfm", "input", "run_codonfm_score")

# Same position, two alternatives: synonymous (Ala -> Ala) and missense (Ala -> Asp)
score_input = CodonFMScoreInput(
    mutations=[
        {"sequence": NATURAL_CDS, "codon_position": 2, "ref_codon": "GCC", "alt_codon": "GCT"},
        {"sequence": NATURAL_CDS, "codon_position": 2, "ref_codon": "GCC", "alt_codon": "GAC"},
    ]
)


In [ ]:
# Display config docs
display_api_reference("codonfm", "config", "run_codonfm_score")

score_config = CodonFMScoreConfig(model_checkpoint=CHECKPOINT, device=DEVICE)


In [ ]:
# Run the score function
score_result = run_codonfm_score(score_input, score_config)


In [ ]:
# Display output docs
display_api_reference("codonfm", "output", "run_codonfm_score")

for r in score_result.results:
    print(f"{r.ref_codon}->{r.alt_codon}  llr={r.llr:+.4f}  (positive = reference favored)")


### `run_codonfm_sample` — propose model-guided codon edits

Sampling masks a subset of codons and refills them from the model's own distribution, drawing only
from the 61 sense codons so a premature stop can never be introduced.

`masking_strategy` chooses which codons to redesign, counted in codons rather than nucleotides.
`fixed_positions` pins codons that must survive — here the terminal stop. Unlike trimming the stop
off the input, pinning keeps it in the sequence the model reads, so the codons before it are scored
in their real context.


In [ ]:
from proto_tools import CodonFMSampleConfig, CodonFMSampleInput, run_codonfm_sample
from proto_tools.transforms.masking import RandomMaskingStrategy


In [ ]:
# Display docs
display_api_reference("codonfm", "input", "run_codonfm_sample")

# Unmasked: the strategy below picks the codons. Mark codons as '___' instead to choose them here.
sample_input = CodonFMSampleInput(sequences=[NATURAL_CDS])


In [ ]:
# Display config docs
display_api_reference("codonfm", "config", "run_codonfm_sample")

stop_codon = len(NATURAL_CDS) // 3  # 1-indexed, so the last codon
sample_config = CodonFMSampleConfig(
    model_checkpoint=CHECKPOINT,
    masking_strategy=RandomMaskingStrategy(num_mutations=3, fixed_positions=[stop_codon]),
    temperature=1.0,
    seed=7,
    device=DEVICE,
)


In [ ]:
# Run the sample function
sample_result = run_codonfm_sample(sample_input, sample_config)


In [ ]:
# Display output docs
display_api_reference("codonfm", "output", "run_codonfm_sample")

sampled = sample_result.results[0].sequence
edited = sum(NATURAL_CDS[i : i + 3] != sampled[i : i + 3] for i in range(0, len(NATURAL_CDS), 3))
print(f"{edited} codons changed; length preserved: {len(sampled) == len(NATURAL_CDS)}")
print(f"terminal stop {NATURAL_CDS[-3:]} preserved: {sampled[-3:] == NATURAL_CDS[-3:]}")


### `run_codonfm_embeddings` — a vector for the whole sequence

Embeddings return Encodon's CLS representation: one fixed-length vector summarizing the coding
sequence, usable as a feature for downstream models or for comparing sequences directly.


In [ ]:
from proto_tools import CodonFMEmbeddingsConfig, CodonFMEmbeddingsInput, run_codonfm_embeddings


In [ ]:
# Display docs
display_api_reference("codonfm", "input", "run_codonfm_embeddings")

embeddings_input = CodonFMEmbeddingsInput(sequences=[NATURAL_CDS])


In [ ]:
# Display config docs
display_api_reference("codonfm", "config", "run_codonfm_embeddings")

embeddings_config = CodonFMEmbeddingsConfig(model_checkpoint=CHECKPOINT, device=DEVICE)


In [ ]:
# Run the embeddings function
embeddings_result = run_codonfm_embeddings(embeddings_input, embeddings_config)


In [ ]:
# Display output docs
display_api_reference("codonfm", "output", "run_codonfm_embeddings")

print(f"CLS embedding dimension: {len(embeddings_result.results[0].embedding)}")


### `run_codonfm_gradient` — differentiable scoring for design loops

The gradient tool takes a *soft* sequence — a per-codon probability matrix rather than fixed
codons — and returns the masked pseudo-log-likelihood plus its gradient with respect to that
matrix. That is what lets CodonFM sit inside a gradient-based design loop.


In [ ]:
from proto_tools import CodonFMGradientConfig, CodonFMGradientInput, run_codonfm_gradient
from proto_tools.tools.masked_models.codonfm import one_hot_codon_logits


In [ ]:
# Display docs
display_api_reference("codonfm", "input", "run_codonfm_gradient")

# Start from the natural CDS as a near-one-hot logit matrix (L x 64)
gradient_input = CodonFMGradientInput(
    inputs=one_hot_codon_logits(NATURAL_CDS, sharpness=2.0), temperature=0.6
)


In [ ]:
# Display config docs
display_api_reference("codonfm", "config", "run_codonfm_gradient")

gradient_config = CodonFMGradientConfig(model_checkpoint=CHECKPOINT, device=DEVICE)


In [ ]:
# Run the gradient function
gradient_result = run_codonfm_gradient(gradient_input, gradient_config)


In [ ]:
# Display output docs
display_api_reference("codonfm", "output", "run_codonfm_gradient")

# One result per candidate; this call passed one, so results holds one
candidate = gradient_result.results[0]

rows, cols = len(candidate.gradient), len(candidate.gradient[0])
print(f"loss: {candidate.loss:.4f}  |  gradient shape: {rows} x {cols}")


---

## Codon optimization, round by round

Everything above is the tool surface. What follows uses those tools together: scoring every
synonymous choice, checking what CodonFM's likelihood already encodes, and then optimizing the
gene one codon at a time while tracking the standard design metrics.


## Standard codon-design metrics

**CAI** (vs a human highly-expressed reference), **MFE** (ViennaRNA), **GC%**, **U%**, and CodonFM **fitness**. The CAI reference below is the relative adaptiveness derived from highly-expressed human genes (GAPDH, ACTB, EEF1A1, ribosomal proteins, ...) fetched from NCBI.

In [ ]:
import math

from proto_tools import ViennaRNAConfig, ViennaRNAInput, run_viennarna

HUMAN_W = {"AAA": 0.378, "AAC": 1.0, "AAG": 1.0, "AAT": 0.5128, "ACA": 0.3571, "ACC": 1.0, "ACG": 0.1071, "ACT": 0.5714, "AGA": 0.3571, "AGC": 0.8409, "AGG": 0.4524, "AGT": 0.2727, "ATA": 0.0246, "ATC": 1.0, "ATG": 1.0, "ATT": 0.4672, "CAA": 0.0732, "CAC": 1.0, "CAG": 1.0, "CAT": 0.7, "CCA": 0.5185, "CCC": 1.0, "CCG": 0.0741, "CCT": 0.7778, "CGA": 0.4524, "CGC": 1.0, "CGG": 0.5952, "CGT": 0.881, "CTA": 0.1028, "CTC": 0.5607, "CTG": 1.0, "CTT": 0.1308, "GAA": 0.4154, "GAC": 1.0, "GAG": 1.0, "GAT": 0.7447, "GCA": 0.2051, "GCC": 1.0, "GCG": 0.1282, "GCT": 0.8034, "GGA": 0.2857, "GGC": 1.0, "GGG": 0.4107, "GGT": 0.4821, "GTA": 0.1171, "GTC": 0.7207, "GTG": 1.0, "GTT": 0.3333, "TAC": 1.0, "TAT": 0.431, "TCA": 0.2727, "TCC": 1.0, "TCG": 0.1818, "TCT": 0.9545, "TGC": 1.0, "TGG": 1.0, "TGT": 0.4286, "TTA": 0.0187, "TTC": 1.0, "TTG": 0.1963, "TTT": 0.5}


def cai(cds):
    lp = []
    for i in range(0, len(cds) - 2, 3):
        c = cds[i : i + 3]; aa = STANDARD_GENETIC_CODE.get(c, "*")
        if aa in ("*", "M", "W"):
            continue
        lp.append(math.log(max(HUMAN_W.get(c, 1e-3), 1e-3)))
    return math.exp(sum(lp) / len(lp)) if lp else float("nan")


def gc(cds): return 100.0 * sum(x in "GC" for x in cds) / len(cds)
def upct(cds): return 100.0 * sum(x in "TU" for x in cds) / len(cds)


def mfe(cds):
    return run_viennarna(ViennaRNAInput(sequences=[cds]), ViennaRNAConfig(temperature=37.0)).results[0].mfe


def codonfm_fitness(cds):
    return run_codonfm_fitness(CodonFMFitnessInput(sequences=[cds]), CodonFMFitnessConfig(model_checkpoint=CHECKPOINT, device=DEVICE)).results[0].fitness


def measure(cds):
    return {"fitness": codonfm_fitness(cds), "CAI": cai(cds), "MFE": mfe(cds), "GC%": gc(cds), "U%": upct(cds)}


natural_metrics = measure(NATURAL_CDS)
print("natural:", {k: round(v, 3) for k, v in natural_metrics.items()})

## Score every synonymous codon choice (one batched call)

For each codon we mask it and ask CodonFM the log-likelihood of every synonymous alternative — a single batched `codonfm-score` request that powers both the correlation below and the round-0 landscape.

In [ ]:
codons = [NATURAL_CDS[i : i + 3] for i in range(0, len(NATURAL_CDS), 3)]

mutations, index = [], []
for pos, c in enumerate(codons, start=1):
    aa = STANDARD_GENETIC_CODE[c]
    if aa == "*" or len(SYNONYMS[aa]) == 1:  # keep the stop; Met/Trp have no synonym
        continue
    for alt in SYNONYMS[aa]:
        mutations.append({"sequence": NATURAL_CDS, "codon_position": pos, "ref_codon": c, "alt_codon": alt})
        index.append((pos, alt))

scored = run_codonfm_score(
    CodonFMScoreInput(mutations=mutations),
    CodonFMScoreConfig(model_checkpoint=CHECKPOINT, device=DEVICE, batch_size=64),
).results
print(f"scored {len(mutations)} synonymous codon choices across {len(codons)} positions")

## CodonFM's prediction already encodes CAI

Does CodonFM rank synonymous codons the way CAI does? Correlate its per-codon log-likelihood against each codon's CAI weight — pooled, and within-family (removing the per-position context so only the *synonymous ranking* remains).

In [ ]:
def pearson(a, b):
    n = len(a); ma = sum(a) / n; mb = sum(b) / n
    cov = sum((x - ma) * (y - mb) for x, y in zip(a, b))
    da = sum((x - ma) ** 2 for x in a) ** 0.5; db = sum((y - mb) ** 2 for y in b) ** 0.5
    return cov / (da * db) if da and db else 0.0


xs = [HUMAN_W[alt] for _, alt in index]
ys = [r.alt_log_likelihood for r in scored]
by_pos = {}
for (pos, alt), r in zip(index, scored):
    by_pos.setdefault(pos, []).append((HUMAN_W[alt], r.alt_log_likelihood))
centered_w, centered_ll = [], []
for pairs in by_pos.values():
    mean_ll = sum(p[1] for p in pairs) / len(pairs)
    for w, ll in pairs:
        centered_w.append(w); centered_ll.append(ll - mean_ll)
agree = sum(1 for p in by_pos.values() if max(p, key=lambda t: t[1])[0] == max(p, key=lambda t: t[0])[0])
r_pool, r_within = pearson(xs, ys), pearson(centered_w, centered_ll)
print(f"Pearson r(CodonFM log-likelihood, CAI weight) = {r_pool:.3f}  (n={len(xs)})")
print(f"  within-family (context-controlled)         = {r_within:.3f}")
print(f"CodonFM top synonymous codon == top-CAI codon at {agree}/{len(by_pos)} positions ({100 * agree // len(by_pos)}%)")
print("=> CodonFM has effectively learned codon-usage bias; its fitness co-varies with CAI (and, below, MFE).")

### Plot — CodonFM log-likelihood vs CAI weight

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))

ax[0].scatter(xs, ys, s=18, alpha=0.5, color="#3b6ea5", edgecolor="none")
m, b = np.polyfit(xs, ys, 1)
xr = np.linspace(min(xs), max(xs), 50)
ax[0].plot(xr, m * xr + b, color="#d1495b", lw=2)
ax[0].set(xlabel="CAI weight of alt codon", ylabel="CodonFM alt log-likelihood",
          title=f"Pooled across all positions  (r = {r_pool:.3f})")

ax[1].scatter(centered_w, centered_ll, s=18, alpha=0.5, color="#2a9d8f", edgecolor="none")
mc, bc = np.polyfit(centered_w, centered_ll, 1)
ax[1].plot(xr, mc * xr + bc, color="#d1495b", lw=2)
ax[1].axhline(0, color="0.7", lw=0.8, ls="--")
ax[1].set(xlabel="CAI weight of alt codon", ylabel="log-likelihood (per-family centered)",
          title=f"Within synonymous family  (r = {r_within:.3f})")

for a in ax:
    a.grid(alpha=0.25)
fig.suptitle("CodonFM's synonymous-codon preference tracks human codon-usage bias", y=1.02, fontsize=12)
fig.tight_layout()
plt.show()

## Iterative CodonFM-guided optimization — one codon per round

Rather than a single greedy pass, we run **coordinate ascent**: each round we re-score every synonymous alternative *in the current sequence context*, apply the single best-improving swap, re-measure all metrics, and repeat until no synonymous swap improves the local codon log-likelihood. Because Encodon is bidirectional, each accepted swap shifts its neighbours' scores — so the optimizer keeps finding new moves over many rounds.

We wrap the whole loop in `ToolInstance.persist()` so the checkpoint is loaded **once** and every round reuses the warm GPU worker.

In [ ]:
import logging
import os

from proto_tools.utils.tool_instance import ToolInstance

# The loop makes ~3 tool calls per round; silence the per-call spinners/logs so we get
# one clean progression line per round instead of ~100 stacked progress bars.
os.environ["PROTO_NO_SPINNER"] = "1"
logging.getLogger("proto_tools.utils.progress").setLevel(logging.WARNING)

MAX_ROUNDS = 60
EPS = 1e-4  # minimum local log-likelihood gain to accept a swap


def score_synonymous(cds):
    """Batched codonfm-score of every synonymous alternative for the current CDS."""
    cur = [cds[i : i + 3] for i in range(0, len(cds), 3)]
    muts, idx = [], []
    for pos, c in enumerate(cur, start=1):
        aa = STANDARD_GENETIC_CODE[c]
        if aa == "*" or len(SYNONYMS[aa]) == 1:
            continue
        for alt in SYNONYMS[aa]:
            if alt == c:
                continue
            muts.append({"sequence": cds, "codon_position": pos, "ref_codon": c, "alt_codon": alt})
            idx.append((pos, alt))
    res = run_codonfm_score(
        CodonFMScoreInput(mutations=muts),
        CodonFMScoreConfig(model_checkpoint=CHECKPOINT, device=DEVICE, batch_size=64),
    ).results
    return cur, idx, res


history = []
change_log = []            # (round, position, old_codon, new_codon, local_gain)
current = NATURAL_CDS

with ToolInstance.persist():                       # warm worker for the whole run
    row = measure(current); row.update(round=0, changed=0, gain=0.0); history.append(row)
    for rnd in range(1, MAX_ROUNDS + 1):
        cur, idx, res = score_synonymous(current)
        best = None                                # (gain, pos, alt)
        for (pos, alt), r in zip(idx, res):
            gain = r.alt_log_likelihood - r.ref_log_likelihood
            if gain > EPS and (best is None or gain > best[0]):
                best = (gain, pos, alt)
        if best is None:
            print(f"converged after {rnd - 1} rounds — no improving synonymous swap remains")
            break
        gain, pos, alt = best
        old = cur[pos - 1]
        cur[pos - 1] = alt
        current = "".join(cur)
        change_log.append((rnd, pos, old, alt, gain))
        row = measure(current); row.update(round=rnd, changed=len(change_log), gain=gain); history.append(row)
        print(f"round {rnd:>2}: codon {pos:>3}  {old}->{alt}  local gain {gain:+.3f}  "
              f"fitness {row['fitness']:.4f}  CAI {row['CAI']:.3f}  MFE {row['MFE']:.1f}")

OPTIMIZED_CDS = current
assert translate(OPTIMIZED_CDS) == PROTEIN         # synonymous only -> identical protein
print(f"\ndone: {len(change_log)} codons changed over {history[-1]['round']} rounds; protein preserved.")

## Progression of every metric across the rounds

Each accepted swap is one point. CodonFM fitness (the objective) rises monotonically; CAI rises and MFE falls **with it**, unprompted — the correlation from above, now playing out over the trajectory.

In [ ]:
rounds = np.array([h["round"] for h in history])
series = {k: np.array([h[k] for h in history]) for k in ("fitness", "CAI", "MFE", "GC%", "U%", "changed")}
titles = {
    "fitness": "CodonFM fitness (mean codon log-lik)",
    "CAI": "CAI (codon adaptation index)",
    "MFE": "MFE  (kcal/mol, ViennaRNA)",
    "GC%": "GC content (%)",
    "U%": "U content (%)",
    "changed": "cumulative codons changed",
}
colors = {"fitness": "#d1495b", "CAI": "#2a9d8f", "MFE": "#3b6ea5",
          "GC%": "#e9a63b", "U%": "#8e6bb0", "changed": "#606060"}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, key in zip(axes.ravel(), titles):
    ax.plot(rounds, series[key], marker="o", ms=4, lw=1.6, color=colors[key])
    ax.scatter([0], [series[key][0]], s=70, color="black", zorder=5, label="natural")
    ax.set(title=titles[key], xlabel="optimization round")
    ax.grid(alpha=0.25)
    ax.legend(loc="best", fontsize=8)
fig.suptitle("Round-by-round codon optimization of human insulin with CodonFM", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## CAI and MFE co-improve with CodonFM fitness

Plotting the trajectory in the (fitness, metric) plane: each point is a round, colour runs early → late. As we climb in CodonFM fitness we drift toward **higher CAI** and **more negative MFE** — the design objectives move together.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sc_kw = dict(c=rounds, cmap="viridis", s=45, zorder=3, edgecolor="white", linewidth=0.4)

ax[0].plot(series["fitness"], series["CAI"], color="0.8", lw=1, zorder=1)
p0 = ax[0].scatter(series["fitness"], series["CAI"], **sc_kw)
ax[0].annotate("natural", (series["fitness"][0], series["CAI"][0]), textcoords="offset points",
               xytext=(6, -12), fontsize=9)
ax[0].annotate("optimized", (series["fitness"][-1], series["CAI"][-1]), textcoords="offset points",
               xytext=(-20, 8), fontsize=9)
ax[0].set(xlabel="CodonFM fitness", ylabel="CAI", title="Fitness vs CAI")

ax[1].plot(series["fitness"], series["MFE"], color="0.8", lw=1, zorder=1)
ax[1].scatter(series["fitness"], series["MFE"], **sc_kw)
ax[1].set(xlabel="CodonFM fitness", ylabel="MFE (kcal/mol)", title="Fitness vs MFE")

for a in ax:
    a.grid(alpha=0.25)
cb = fig.colorbar(p0, ax=ax, fraction=0.045, pad=0.02)
cb.set_label("optimization round")
fig.suptitle("Objectives move together along the optimization trajectory", y=1.0, fontsize=12)
plt.show()

## Convergence — per-round gain and where the codons changed

Left: the local log-likelihood gain accepted each round decays as the easy wins are used up (why the run terminates). Right: the positions along the CDS that were rewritten, coloured by the round they changed.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))

gains = np.array([h["gain"] for h in history[1:]])
gr = np.arange(1, len(gains) + 1)
ax[0].bar(gr, gains, color="#3b6ea5", alpha=0.85)
ax[0].set(xlabel="optimization round", ylabel="accepted local log-lik gain",
          title="Diminishing returns per round")
ax[0].grid(alpha=0.25, axis="y")

if change_log:
    rnd_arr = np.array([c[0] for c in change_log])
    pos_arr = np.array([c[1] for c in change_log])
    sc = ax[1].scatter(pos_arr, rnd_arr, c=rnd_arr, cmap="viridis", s=55, edgecolor="white", linewidth=0.4)
    for _r, _p, _o, _n, _g in change_log:
        ax[1].annotate(f"{_o}>{_n}", (_p, _r), textcoords="offset points", xytext=(0, 6),
                       ha="center", fontsize=6, color="0.35")
    ax[1].set(xlabel="codon position along CDS", ylabel="round changed",
              title="Which codons were rewritten", xlim=(0, len(codons) + 1))
    ax[1].grid(alpha=0.25)
    fig.colorbar(sc, ax=ax[1], fraction=0.045, pad=0.02, label="round")
fig.tight_layout()
plt.show()

## Before vs after — natural, then CodonFM-optimized

Same protein, rewritten codons. The optimizer pushed CodonFM fitness and CAI up and MFE down together.

In [ ]:
opt_metrics = {k: series[k][-1] for k in ("fitness", "CAI", "MFE", "GC%", "U%")}
print(f"{'metric':<9}{'natural':>12}{'optimized':>12}{'delta':>12}")
for k in ("fitness", "CAI", "MFE", "GC%", "U%"):
    d = opt_metrics[k] - natural_metrics[k]
    print(f"{k:<9}{natural_metrics[k]:>12.3f}{opt_metrics[k]:>12.3f}{d:>+12.3f}")
n_changed = sum(a != b for a, b in zip(codons, [OPTIMIZED_CDS[i:i+3] for i in range(0, len(OPTIMIZED_CDS), 3)]))
print(f"\ncodons changed: {n_changed}/{len(codons)}   protein identical: {translate(OPTIMIZED_CDS) == PROTEIN}")

# normalized grouped bar (each metric scaled to its natural value so they share an axis)
keys = ["fitness", "CAI", "MFE", "GC%", "U%"]
nat = np.array([natural_metrics[k] for k in keys])
opt = np.array([opt_metrics[k] for k in keys])
norm_nat = np.ones(len(keys))
norm_opt = opt / nat
x = np.arange(len(keys))
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.bar(x - 0.2, norm_nat, 0.4, label="natural", color="0.7")
ax.bar(x + 0.2, norm_opt, 0.4, label="optimized", color="#d1495b")
ax.axhline(1.0, color="0.5", lw=0.8, ls="--")
for xi, (n, o) in enumerate(zip(nat, opt)):
    ax.annotate(f"{n:.2f}", (xi - 0.2, 1.0), ha="center", va="bottom", fontsize=8)
    ax.annotate(f"{o:.2f}", (xi + 0.2, o / n), ha="center", va="bottom", fontsize=8)
ax.set(xticks=x, xticklabels=keys, ylabel="value / natural value",
       title="Natural vs CodonFM-optimized insulin CDS (normalized to natural)")
ax.legend()
ax.grid(alpha=0.25, axis="y")
fig.tight_layout()
plt.show()

---
**Takeaway.** CodonFM's masked codon likelihood is not an independent objective bolted onto CAI/MFE — it has *learned* human codon-usage bias (the correlation plots), so optimizing it round-by-round drags CAI up and MFE down for free. The bidirectional context is what makes the iterative, re-scored loop worthwhile: each swap reshapes the landscape for the next.